# fase_5 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 5.

**Purpose**: Migrasi data Rapor dan Penilaian dengan Mapping Kolom Spesifik

In [ ]:
import sys
import os
import mysql.connector
import pandas as pd
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [ ]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

## 2. Ambil Data dari DB Lama

In [ ]:
hanif_tables_map = [
    ('format_rapor', 'rapor_format'),
    ('format_rapor_detil', 'rapor_format_sub'),
    ('format_rapor_rumus', 'rapor_format_formula'),
    ('format_rapor_detil_rumus', 'rapor_format_formula_sub'),
    ('format_raport_level', 'rapor_level_config'),
    ('rapor', 'rapor_siswa'),
    ('file_rapor_siswa', 'rapor_siswa_file'),
    ('history_rapor', 'rapor_lacak')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    try:
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()
        print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")
    except Exception as e:
        print(f"❌ ERROR loading {old_t}: {e}")

## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [ ]:
transformed_dfs = {}

# 1. format_rapor -> rapor_format
if 'format_rapor' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor'])
    mapping = {
        'idformat_rapor': 'id_rapor_format',
        'idpendkursus': 'id_kursus', 'title': 'judul_rapor'
    }
    transformed_dfs['rapor_format'] = df.rename(columns=mapping)[list(mapping.values())]

# 2. format_rapor_detil -> rapor_format_sub
if 'format_rapor_detil' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_detil'])
    mapping = {
        'idformat_rd': 'id_rapor_format_sub',
        'idformat_rapor': 'id_rapor_format', 'subtitle': 'sub_judul_rapor'
    }
    transformed_dfs['rapor_format_sub'] = df.rename(columns=mapping)[list(mapping.values())]

# 3. format_rapor_rumus -> rapor_format_formula
if 'format_rapor_rumus' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_rumus'])
    mapping = {
        'idfrr': 'id_rapor_format_formula',
        'idformat_rapor': 'id_rapor_format', 'param_operator': 'logika_operator'
    }
    transformed_dfs['rapor_format_formula'] = df.rename(columns=mapping)[list(mapping.values())]

# 4. format_rapor_detil_rumus -> rapor_format_formula_sub
if 'format_rapor_detil_rumus' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_detil_rumus'])
    mapping = {
        'idfrdr': 'id_rapor_format_formula_sub',
        'idformat_rd': 'id_rapor_format_sub', 'param_operator': 'logika_operator',
        'idlevel': 'id_level'
    }
    transformed_dfs['rapor_format_formula_sub'] = df.rename(columns=mapping)[list(mapping.values())]

# 5. format_raport_level -> rapor_level_config
if 'format_raport_level' in raw_data:
    df = pd.DataFrame(raw_data['format_raport_level'])
    mapping = {
        'idformat_rl': 'id_rapor_level_config', 'idlevel': 'id_level',
        'idpendkursus': 'id_kursus', 'idformat_rapor': 'id_rapor_format'
    }
    transformed_dfs['rapor_level_config'] = df.rename(columns=mapping)[list(mapping.values())]

# 6. rapor_sub_level (Source: -)
transformed_dfs['rapor_sub_level'] = pd.DataFrame()

# 7. rapor -> rapor_siswa
if 'rapor' in raw_data:
    df = pd.DataFrame(raw_data['rapor'])
    mapping = {
        'idrapor': 'id_rapor_siswa', 'idjadwal': 'id_jadwal', 'idsiswa': 'id_siswa',
        'tanggal': 'tanggal_input', 'idp_nilai': 'id_parameter_nilai', 'nilai': 'final_result'
    }
    transformed_dfs['rapor_siswa'] = df.rename(columns=mapping)[list(mapping.values())]

# 8. file_rapor_siswa -> rapor_siswa_file
if 'file_rapor_siswa' in raw_data:
    df = pd.DataFrame(raw_data['file_rapor_siswa'])
    mapping = {
        'idfile': 'id_rapor_siswa_file', 'idsiswa': 'id_rapor_siswa', 'path': 'file_rapor_path'
    }
    transformed_dfs['rapor_siswa_file'] = df.rename(columns=mapping)[list(mapping.values())]

# 9. history_rapor -> rapor_lacak
if 'history_rapor' in raw_data:
    df = pd.DataFrame(raw_data['history_rapor'])
    mapping = {
        'idhistori': 'id_rapor_lacak', 'idsiswa': 'id_siswa',
        'idjadwal': 'id_jadwal', 'tgl': 'tanggal_terkirim', 'status': 'status_pengiriman'
    }
    df = df.rename(columns=mapping)
    df['id_rapor_siswa_file'] = None
    transformed_dfs['rapor_lacak'] = df[list(mapping.values()) + ['id_rapor_siswa_file']]

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 5 selesai.")

## 4. Export ke Pickle

In [ ]:
file_name = 'fase_5_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

total_records = sum(len(df) for df in transformed_dfs.values())
migration_result = {
    'fase': 'fase_5',
    'script': 'script_hanif',
    'fase_num': 5,
    'status': 'ready_for_insert',
    'records_transformed': total_records,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()